# 08 — Reorder Prediction

## 1. Objective

The objective of this notebook is to train machine learning models that predict whether a candidate product will be reordered in a customer's next basket.

The target variable is:

- `1` — the product appears in the target order
- `0` — the product does not appear in the target order

Because only about 9.78% of observations belong to the positive class, we will evaluate the models using metrics such as Precision, Recall, F1-score, PR AUC, and ROC AUC rather than relying only on Accuracy.

In [0]:
from pyspark.sql import functions as F

features_df = spark.table("workspace.ml_data.reorder_features")

display(features_df.limit(10))

user_id,target_order_id,product_id,target_reordered,customer_prior_orders,customer_total_products,customer_unique_products,customer_avg_basket_size,customer_reorder_rate,customer_avg_days_between_orders,customer_std_days_between_orders,customer_avg_order_hour,customer_active_days_of_week,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend,customer_preferred_dow,customer_preferred_day_share,customer_preferred_hour,customer_preferred_hour_share,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position,product_name,aisle_id,department_id,product_purchases_per_customer,product_order_share,product_preferred_dow,product_preferred_day_share,product_preferred_hour,product_preferred_hour_share,user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum,user_product_max_streak,user_product_current_streak,customer_aisle_affinity,customer_department_affinity,aisle_in_last_basket,department_in_last_basket,customer_basket_stability,repeat_stability_signal,last_basket_context_level,stability_weighted_context,target_order_dow,target_order_hour,customer_day_match,product_day_match,customer_day_distance,product_day_distance,customer_hour_distance_circular,product_hour_distance_circular,target_days_since_prior_order,order_timing_deviation_days,order_timing_ratio,expected_basket_size,recent_vs_usual_basket_ratio,target_order_number,has_reorder_cycle,has_order_timing_ratio
197149,1575521,41593,0,17,248,92,14.59,0.629,15.81,9.97,13.71,7,7,7.33,-7.26,1,0.2941,6,0.1176,4018,2416,0.3987,10.12,Blue Cheese Crumbles,21,16,1.66,0.001963,0,0.2282,15,0.0963,2,0.5,3.0,13,15,0.1176,3,2.0,1.5,0,1,2,0.3333,0.4,0.2157,1,0,0.0605,0.2742,0,1,1.0,0.0,1,0.3333,4,18,0,0,3,3,12,3,30.0,14.19,1.9,10.96,0.5,18,1,1
93315,1823910,46226,1,17,217,90,12.76,0.5853,17.88,9.37,14.18,7,11,15.33,2.57,1,0.2353,12,0.2941,4892,1821,0.6278,9.04,Thick & Crispy Tortilla Chips,107,19,2.69,0.002389,0,0.1995,11,0.0901,3,0.6667,11.33,13,17,0.1765,1,2.0,0.5,1,2,3,0.6667,0.6,0.4902,1,1,0.0138,0.0922,1,1,0.3333,0.3333,3,0.3333,1,11,1,0,0,1,1,0,15.0,-2.88,0.84,14.05,1.2,18,1,1
139432,1891550,24086,0,51,946,208,18.55,0.7801,7.0,3.27,11.06,7,21,26.67,8.12,0,0.4706,8,0.2549,407,141,0.6536,7.33,Beef Tenderloin Steak,122,12,2.89,1.99E-4,0,0.2064,13,0.0983,2,0.5,21.5,4,5,0.0392,47,1.0,47.0,0,0,0,0.0,0.0,-0.0392,2,0,0.0085,0.0296,0,0,0.122,0.0,0,0.0,5,11,0,0,2,2,3,2,12.0,5.0,1.71,22.61,1.44,52,1,1
24065,837103,4390,0,42,603,157,14.36,0.7396,5.59,3.82,15.93,7,9,6.67,-7.69,5,0.3095,17,0.2381,482,305,0.3672,9.39,Calcium Plus Vitamin D Vegetable Oil Spread,36,16,1.58,2.35E-4,0,0.2116,13,0.1058,3,0.6667,6.67,1,19,0.0714,24,9.0,2.67,0,0,0,0.0,0.0,-0.0714,1,0,0.0216,0.2305,0,0,0.1667,0.0,0,0.0,3,18,0,0,2,3,1,5,5.0,-0.59,0.89,10.52,0.46,43,1,1
171369,1728184,26856,0,34,610,141,17.94,0.7689,9.94,6.9,12.79,7,15,14.33,-3.61,0,0.3235,12,0.1471,3428,1301,0.6205,7.98,Organic Ezekiel 4:9 Sesame Bread,112,3,2.63,0.001674,1,0.1969,11,0.0869,6,0.8333,14.83,19,32,0.1765,3,2.6,1.15,0,1,3,0.3333,0.6,0.1568,3,0,0.0098,0.023,0,0,0.2083,0.0,0,0.0,1,16,0,1,1,0,4,5,7.0,-2.94,0.7,16.14,0.8,35,1,1
88619,528541,9006,0,33,218,116,6.61,0.4679,9.88,8.71,14.33,7,3,2.33,-4.28,1,0.303,12,0.1515,1180,567,0.5195,8.55,Naturе's Calorie-Free Sweetener,17,13,2.08,5.76E-4,1,0.2051,10,0.0975,1,0.0,5.0,8,8,0.0303,26,0.0,0.0,0,0,0,0.0,0.0,-0.0303,1,0,0.0275,0.0642,0,0,0.25,0.0,0,0.0,1,12,1,1,0,0,0,2,4.0,-5.88,0.4,4.47,0.35,34,0,1
32324,1039313,9990,0,16,175,78,10.94,0.5543,16.8,10.01,11.63,4,22,12.0,1.06,0,0.4375,9,0.1875,135,44,0.6741,7.95,Gluten Free Classy Slice Bread,112,3,3.07,6.6E-5,0,0.2,13,0.1333,2,0.5,9.0,7,12,0.125,5,5.0,1.0,0,0,1,0.0,0.2,-0.

## 2. Train–Validation Split

Each customer appears several times because they have multiple candidate products.

Therefore, we split the dataset **by customer**, not randomly by individual rows.

This ensures that the same customer cannot appear in both the training and validation datasets, giving us a more realistic evaluation of the model on unseen customers.

We use:

- **80% of customers** for training
- **20% of customers** for validation

Customer A → TRAIN only

Customer B → TRAIN only

Customer C → VALIDATION only

In [0]:
# Get one row per customer
users_df = features_df.select("user_id").distinct()

# Split customers into 80% training and 20% validation
train_users_df, validation_users_df = users_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Create the training dataset
train_df = (
    features_df
    .join(train_users_df, on="user_id", how="inner")
)

# Create the validation dataset
validation_df = (
    features_df
    .join(validation_users_df, on="user_id", how="inner")
)

print("Train customers:", train_users_df.count())
print("Validation customers:", validation_users_df.count())

print("Train rows:", train_df.count())
print("Validation rows:", validation_df.count())

Train customers: 104849
Validation customers: 26360
Train rows: 6785209
Validation rows: 1689452


### 2.1 — Validate the Customer Split

We verify that no customer appears in both the training and validation datasets.

A result of `0` confirms that the two datasets are completely separated by customer.

In [0]:
customer_overlap = (
    train_users_df
    .join(
        validation_users_df,
        on="user_id",
        how="inner"
    )
    .count()
)

print("Customers appearing in both sets:", customer_overlap)

Customers appearing in both sets: 20809


### 2.2 — Check the Target Distribution

We check the proportion of reordered and non-reordered products in both datasets.

The training and validation sets should have similar class distributions so that the validation set remains representative of the original data.

In [0]:
# Total number of rows in each dataset
train_total = train_df.count()
validation_total = validation_df.count()

# Target distribution in training data
train_distribution_df = (
    train_df
    .groupBy("target_reordered")
    .count()
    .withColumn("dataset", F.lit("Train"))
    .withColumn(
        "percentage",
        F.round(F.col("count") / train_total * 100, 2)
    )
)

# Target distribution in validation data
validation_distribution_df = (
    validation_df
    .groupBy("target_reordered")
    .count()
    .withColumn("dataset", F.lit("Validation"))
    .withColumn(
        "percentage",
        F.round(F.col("count") / validation_total * 100, 2)
    )
)

# Combine both results
target_distribution_df = (
    train_distribution_df
    .unionByName(validation_distribution_df)
    .select(
        "dataset",
        "target_reordered",
        "count",
        "percentage"
    )
    .orderBy("dataset", "target_reordered")
)

display(target_distribution_df)

dataset,target_reordered,count,percentage
Train,0,6120934,90.21
Train,1,664275,9.79
Validation,0,1524903,90.26
Validation,1,164549,9.74


## 3. Prepare the Model Features

Not every column in the dataset should be used directly by the machine learning model.

We exclude:

- Customer, order, and product IDs because they are identifiers.
- `product_name` because it is text metadata.
- `target_reordered` because it is the prediction target.

`aisle_id` and `department_id` are categorical variables and will be encoded separately.

All remaining columns will be treated as numerical features.

In [0]:
# Columns that should not be used as model features
excluded_columns = [
    "user_id",
    "target_order_id",
    "product_id",
    "product_name",
    "target_reordered"
]

# Categorical features
categorical_columns = [
    "aisle_id",
    "department_id"
]

# Numerical features
numerical_columns = [
    column
    for column in features_df.columns
    if column not in excluded_columns + categorical_columns
]

print("Numerical features:", len(numerical_columns))
print("Categorical features:", len(categorical_columns))

print("\nCategorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

Numerical features: 67
Categorical features: 2

Categorical columns:
['aisle_id', 'department_id']

Numerical columns:
['customer_prior_orders', 'customer_total_products', 'customer_unique_products', 'customer_avg_basket_size', 'customer_reorder_rate', 'customer_avg_days_between_orders', 'customer_std_days_between_orders', 'customer_avg_order_hour', 'customer_active_days_of_week', 'customer_last_basket_size', 'customer_avg_last_3_basket_size', 'customer_basket_size_trend', 'customer_preferred_dow', 'customer_preferred_day_share', 'customer_preferred_hour', 'customer_preferred_hour_share', 'product_purchase_count', 'product_unique_customers', 'product_reorder_rate', 'product_avg_cart_position', 'product_purchases_per_customer', 'product_order_share', 'product_preferred_dow', 'product_preferred_day_share', 'product_preferred_hour', 'product_preferred_hour_share', 'user_product_order_count', 'user_product_reorder_rate', 'user_product_avg_cart_position', 'user_product_first_order', 'user_p

### 3.1 — Encode Categorical Features

`aisle_id` and `department_id` represent categories rather than numerical quantities.

We therefore:

1. Convert each category into an indexed value.
2. Apply one-hot encoding so the model does not interpret category IDs as numerical magnitudes.

The encoders are fitted only on the training data to avoid using information from the validation set during training.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Convert categorical IDs into category indexes
aisle_indexer = StringIndexer(
    inputCol="aisle_id",
    outputCol="aisle_index",
    handleInvalid="keep"
)

department_indexer = StringIndexer(
    inputCol="department_id",
    outputCol="department_index",
    handleInvalid="keep"
)

# One-hot encode the indexed categories
encoder = OneHotEncoder(
    inputCols=["aisle_index", "department_index"],
    outputCols=["aisle_ohe", "department_ohe"],
    handleInvalid="keep"
)

# Build the categorical preprocessing pipeline
categorical_pipeline = Pipeline(
    stages=[
        aisle_indexer,
        department_indexer,
        encoder
    ]
)

# Fit only on training data
categorical_model = categorical_pipeline.fit(train_df)

# Apply the same transformation to both datasets
train_encoded_df = categorical_model.transform(train_df)
validation_encoded_df = categorical_model.transform(validation_df)

display(
    train_encoded_df.select(
        "aisle_id",
        "aisle_index",
        "aisle_ohe",
        "department_id",
        "department_index",
        "department_ohe"
    ).limit(10)
)

aisle_id,aisle_index,aisle_ohe,department_id,department_index,department_ohe
4,43.0,"{""type"":""0"",""size"":""135"",""indices"":[""43""],""values"":[""1.0""]}",9,7.0,"{""type"":""0"",""size"":""22"",""indices"":[""7""],""values"":[""1.0""]}"
81,19.0,"{""type"":""0"",""size"":""135"",""indices"":[""19""],""values"":[""1.0""]}",15,6.0,"{""type"":""0"",""size"":""22"",""indices"":[""6""],""values"":[""1.0""]}"
106,31.0,"{""type"":""0"",""size"":""135"",""indices"":[""31""],""values"":[""1.0""]}",12,12.0,"{""type"":""0"",""size"":""22"",""indices"":[""12""],""values"":[""1.0""]}"
59,32.0,"{""type"":""0"",""size"":""135"",""indices"":[""32""],""values"":[""1.0""]}",15,6.0,"{""type"":""0"",""size"":""22"",""indices"":[""6""],""values"":[""1.0""]}"
41,88.0,"{""type"":""0"",""size"":""135"",""indices"":[""88""],""values"":[""1.0""]}",8,18.0,"{""type"":""0"",""size"":""22"",""indices"":[""18""],""values"":[""1.0""]}"
83,0.0,"{""type"":""0"",""size"":""135"",""indices"":[""0""],""values"":[""1.0""]}",4,0.0,"{""type"":""0"",""size"":""22"",""indices"":[""0""],""values"":[""1.0""]}"
116,7.0,"{""type"":""0"",""size"":""135"",""indices"":[""7""],""values"":[""1.0""]}",1,4.0,"{""type"":""0"",""size"":""22"",""indices"":[""4""],""values"":[""1.0""]}"
16,16.0,"{""type"":""0"",""size"":""135"",""indices"":[""16""],""values"":[""1.0""]}",4,0.0,"{""type"":""0"",""size"":""22"",""indices"":[""0""],""values"":[""1.0""]}"
74,60.0,"{""type"":""0"",""size"":""135"",""indices"":[""60""],""values"":[""1.0""]}",17,8.0,"{""type"":""0"",""size"":""22"",""indices"":[""8""],""values"":[""1.0""]}"
83,0.0,"{""type"":""0"",""size"":""135"",""indices"":[""0""],""values"":[""1.0""]}",4,0.0,"{""type"":""0"",""size"":""22"",""indices"":[""0""],""values"":[""1.0""]}"


### 3.2 — Assemble the Final Feature Vector

Spark ML requires all predictor variables to be combined into a single feature vector.

We combine the numerical features with the encoded aisle and department features into one column called `features`.

In [0]:
from pyspark.ml.feature import VectorAssembler

# All inputs used by the model
model_input_columns = (
    numerical_columns
    + ["aisle_ohe", "department_ohe"]
)

# Combine all features into one vector
assembler = VectorAssembler(
    inputCols=model_input_columns,
    outputCol="features",
    handleInvalid="keep"
)

# Apply to training and validation data
train_model_df = assembler.transform(train_encoded_df)

validation_model_df = assembler.transform(validation_encoded_df)

# Keep the main columns needed for modeling
display(
    train_model_df.select(
        "user_id",
        "product_id",
        "target_order_id",
        "target_reordered",
        "features"
    ).limit(5)
)

user_id,product_id,target_order_id,target_reordered,features
123467,43483,660342,0,"{""type"":""0"",""size"":""224"",""indices"":[""0"",""1"",""2"",""3"",""4"",""5"",""7"",""8"",""9"",""10"",""11"",""13"",""14"",""15"",""16"",""17"",""18"",""19"",""20"",""21"",""22"",""23"",""24"",""25"",""26"",""27"",""28"",""29"",""30"",""31"",""32"",""33"",""34"",""35"",""36"",""37"",""38"",""39"",""40"",""41"",""42"",""43"",""44"",""45"",""46"",""47"",""48"",""49"",""50"",""51"",""52"",""55"",""56"",""57"",""58"",""59"",""61"",""62"",""63"",""64"",""65"",""66"",""110"",""209""],""values"":[""4.0"",""69.0"",""55.0"",""17.25"",""0.2029"",""30.0"",""16.5"",""3.0"",""17.0"",""19.0"",""1.75"",""0.5"",""12.0"",""0.25"",""788.0"",""423.0"",""0.4632"",""9.1"",""1.86"",""3.85E-4"",""1.0"",""0.1827"",""15.0"",""0.0952"",""2.0"",""0.5"",""8.0"",""2.0"",""4.0"",""0.5"",""1.0"",""2.0"",""0.5"",""1.0"",""2.0"",""2.0"",""0.6667"",""0.5"",""0.1667"",""1.0"",""1.0"",""0.0435"",""0.0435"",""1.0"",""1.0"",""0.1724"",""0.1724"",""3.0"",""0.1724"",""4.0"",""21.0"",""3.0"",""3.0"",""9.0"",""6.0"",""30.0"",""1.0"",""18.13"",""1.1"",""5.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
203322,22007,989667,0,"{""type"":""0"",""size"":""224"",""indices"":[""0"",""1"",""2"",""3"",""4"",""5"",""6"",""7"",""8"",""9"",""10"",""11"",""13"",""14"",""15"",""16"",""17"",""18"",""19"",""20"",""21"",""23"",""24"",""25"",""26"",""28"",""29"",""30"",""31"",""32"",""37"",""39"",""40"",""41"",""43"",""44"",""51"",""52"",""55"",""56"",""57"",""58"",""59"",""60"",""61"",""62"",""63"",""64"",""66"",""86"",""208""],""values"":[""5.0"",""60.0"",""49.0"",""12.0"",""0.1833"",""24.75"",""8.53"",""13.4"",""3.0"",""13.0"",""12.67"",""0.67"",""0.6"",""13.0"",""0.4"",""1523.0"",""1201.0"",""0.2114"",""9.42"",""1.27"",""7.44E-4"",""0.2088"",""11.0"",""0.0906"",""1.0"",""5.0"",""1.0"",""1.0"",""0.2"",""5.0"",""1.0"",""0.2"",""-0.2"",""1.0"",""0.0167"",""0.0667"",""1.0"",""9.0"",""1.0"",""1.0"",""4.0"",""2.0"",""30.0"",""5.25"",""1.21"",""12.34"",""1.06"",""6.0"",""1.0"",""1.0"",""1.0""]}"
87923,33279,123045,0,"{""type"":""0"",""size"":""224"",""indices"":[""0"",""1"",""2"",""3"",""4"",""5"",""7"",""8"",""9"",""10"",""11"",""13"",""14"",""15"",""16"",""17"",""18"",""19"",""20"",""21"",""23"",""24"",""25"",""26"",""27"",""28"",""29"",""30"",""31"",""32"",""33"",""34"",""35"",""36"",""37"",""38"",""39"",""40"",""41"",""42"",""43"",""44"",""45"",""46"",""47"",""48"",""49"",""50"",""52"",""53"",""54"",""57"",""58"",""59"",""61"",""62"",""63"",""64"",""65"",""66"",""98"",""214""],""values"":[""4.0"",""12.0"",""11.0"",""3.0"",""0.0833"",""30.0"",""12.75"",""4.0"",""3.0"",""2.33"",""-0.67"",""0.25"",""10.0"",""0.25"",""3104.0"",""1565.0"",""0.4958"",""7.41"",""1.98"",""0.001516"",""0.1859"",""15.0"",""0.0921"",""2.0"",""0.5"",""1.5"",""3.0"",""4.0"",""0.5"",""1.0"",""1.0"",""1.0"",""1.0"",""2.0"",""2.0"",""0.6667"",""0.5"",""0.1667"",""2.0"",""2.0"",""0.1667"",""0.1667"",""1.0"",""1.0"",""0.2"",""0.2"",""3.0"",""0.2"",""23.0"",""1.0"",""1.0"",""11.0"",""8.0"",""30.0"",""1.0"",""2.67"",""0.78"",""5.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
20907,26668,187518,0,"{""type"":""0"",""size"":""224"",""indices"":[""0"",""1"",""2"",""3"",""4"",""5"",""6"",""7"",""8"",""9"",""10"",""11"",""13"",""14"",""15"",""16"",""17"",""18"",""19"",""20"",""21"",""23"",""24"",""25"",""26"",""28"",""29"",""30"",""31"",""32"",""40"",""41"",""43"",""44"",""51"",""52"",""55"",""56"",""57"",""58"",""59"",""60"",""61"",""62"",""63"",""64"",""66"",""99"",""208""],""values"":[""31.0"",""181.0"",""83.0"",""5.84"",""0.5414"",""10.6"",""8.65"",""13.9"",""7.0"",""2.0"",""2.0"",""-3.84"",""0.2581"",""11.0"",""0.1935"",""3172.0"",""2019.0"",""0.3635"",""11.17"",""1.57"",""0.001549"",""0.238"",""11.0"",""0.0936"",""1.0"",""10.0"",""1.0"",""1.0"",""0.0323"",""31.0"",""-0.0323"",""1.0"",""0.011"",""0.0663"",""3.0"",""17.0"",""3.0"",""3.0"",""6.0"",""6.0"",""3.0"",""-7.6"",""0.28"",""3.92"",""0.34"",""32.0"",""1.0"",""1.0"",""1.0""]}"
14578,7076,237091,0

### 3.3 — Prepare the Training and Validation Datasets

We prepare compact datasets for machine learning.

The prediction target `target_reordered` is renamed to `label`, while the identifiers are kept only so we can later analyze predictions and generate product recommendations.

In [0]:
train_ml_df = (
    train_model_df
    .select(
        "user_id",
        "target_order_id",
        "product_id",
        F.col("target_reordered").cast("double").alias("label"),
        "features"
    )
)

validation_ml_df = (
    validation_model_df
    .select(
        "user_id",
        "target_order_id",
        "product_id",
        F.col("target_reordered").cast("double").alias("label"),
        "features"
    )
)

display(train_ml_df.limit(5))

user_id,target_order_id,product_id,label,features
123467,660342,43483,0.0,"{""type"":""0"",""size"":""224"",""indices"":[""0"",""1"",""2"",""3"",""4"",""5"",""7"",""8"",""9"",""10"",""11"",""13"",""14"",""15"",""16"",""17"",""18"",""19"",""20"",""21"",""22"",""23"",""24"",""25"",""26"",""27"",""28"",""29"",""30"",""31"",""32"",""33"",""34"",""35"",""36"",""37"",""38"",""39"",""40"",""41"",""42"",""43"",""44"",""45"",""46"",""47"",""48"",""49"",""50"",""51"",""52"",""55"",""56"",""57"",""58"",""59"",""61"",""62"",""63"",""64"",""65"",""66"",""110"",""209""],""values"":[""4.0"",""69.0"",""55.0"",""17.25"",""0.2029"",""30.0"",""16.5"",""3.0"",""17.0"",""19.0"",""1.75"",""0.5"",""12.0"",""0.25"",""788.0"",""423.0"",""0.4632"",""9.1"",""1.86"",""3.85E-4"",""1.0"",""0.1827"",""15.0"",""0.0952"",""2.0"",""0.5"",""8.0"",""2.0"",""4.0"",""0.5"",""1.0"",""2.0"",""0.5"",""1.0"",""2.0"",""2.0"",""0.6667"",""0.5"",""0.1667"",""1.0"",""1.0"",""0.0435"",""0.0435"",""1.0"",""1.0"",""0.1724"",""0.1724"",""3.0"",""0.1724"",""4.0"",""21.0"",""3.0"",""3.0"",""9.0"",""6.0"",""30.0"",""1.0"",""18.13"",""1.1"",""5.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
203322,989667,22007,0.0,"{""type"":""0"",""size"":""224"",""indices"":[""0"",""1"",""2"",""3"",""4"",""5"",""6"",""7"",""8"",""9"",""10"",""11"",""13"",""14"",""15"",""16"",""17"",""18"",""19"",""20"",""21"",""23"",""24"",""25"",""26"",""28"",""29"",""30"",""31"",""32"",""37"",""39"",""40"",""41"",""43"",""44"",""51"",""52"",""55"",""56"",""57"",""58"",""59"",""60"",""61"",""62"",""63"",""64"",""66"",""86"",""208""],""values"":[""5.0"",""60.0"",""49.0"",""12.0"",""0.1833"",""24.75"",""8.53"",""13.4"",""3.0"",""13.0"",""12.67"",""0.67"",""0.6"",""13.0"",""0.4"",""1523.0"",""1201.0"",""0.2114"",""9.42"",""1.27"",""7.44E-4"",""0.2088"",""11.0"",""0.0906"",""1.0"",""5.0"",""1.0"",""1.0"",""0.2"",""5.0"",""1.0"",""0.2"",""-0.2"",""1.0"",""0.0167"",""0.0667"",""1.0"",""9.0"",""1.0"",""1.0"",""4.0"",""2.0"",""30.0"",""5.25"",""1.21"",""12.34"",""1.06"",""6.0"",""1.0"",""1.0"",""1.0""]}"
87923,123045,33279,0.0,"{""type"":""0"",""size"":""224"",""indices"":[""0"",""1"",""2"",""3"",""4"",""5"",""7"",""8"",""9"",""10"",""11"",""13"",""14"",""15"",""16"",""17"",""18"",""19"",""20"",""21"",""23"",""24"",""25"",""26"",""27"",""28"",""29"",""30"",""31"",""32"",""33"",""34"",""35"",""36"",""37"",""38"",""39"",""40"",""41"",""42"",""43"",""44"",""45"",""46"",""47"",""48"",""49"",""50"",""52"",""53"",""54"",""57"",""58"",""59"",""61"",""62"",""63"",""64"",""65"",""66"",""98"",""214""],""values"":[""4.0"",""12.0"",""11.0"",""3.0"",""0.0833"",""30.0"",""12.75"",""4.0"",""3.0"",""2.33"",""-0.67"",""0.25"",""10.0"",""0.25"",""3104.0"",""1565.0"",""0.4958"",""7.41"",""1.98"",""0.001516"",""0.1859"",""15.0"",""0.0921"",""2.0"",""0.5"",""1.5"",""3.0"",""4.0"",""0.5"",""1.0"",""1.0"",""1.0"",""1.0"",""2.0"",""2.0"",""0.6667"",""0.5"",""0.1667"",""2.0"",""2.0"",""0.1667"",""0.1667"",""1.0"",""1.0"",""0.2"",""0.2"",""3.0"",""0.2"",""23.0"",""1.0"",""1.0"",""11.0"",""8.0"",""30.0"",""1.0"",""2.67"",""0.78"",""5.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
20907,187518,26668,0.0,"{""type"":""0"",""size"":""224"",""indices"":[""0"",""1"",""2"",""3"",""4"",""5"",""6"",""7"",""8"",""9"",""10"",""11"",""13"",""14"",""15"",""16"",""17"",""18"",""19"",""20"",""21"",""23"",""24"",""25"",""26"",""28"",""29"",""30"",""31"",""32"",""40"",""41"",""43"",""44"",""51"",""52"",""55"",""56"",""57"",""58"",""59"",""60"",""61"",""62"",""63"",""64"",""66"",""99"",""208""],""values"":[""31.0"",""181.0"",""83.0"",""5.84"",""0.5414"",""10.6"",""8.65"",""13.9"",""7.0"",""2.0"",""2.0"",""-3.84"",""0.2581"",""11.0"",""0.1935"",""3172.0"",""2019.0"",""0.3635"",""11.17"",""1.57"",""0.001549"",""0.238"",""11.0"",""0.0936"",""1.0"",""10.0"",""1.0"",""1.0"",""0.0323"",""31.0"",""-0.0323"",""1.0"",""0.011"",""0.0663"",""3.0"",""17.0"",""3.0"",""3.0"",""6.0"",""6.0"",""3.0"",""-7.6"",""0.28"",""3.92"",""0.34"",""32.0"",""1.0"",""1.0"",""1.0""]}"
14578,237091,7076,0.0,

user_id           → kept for later recommendation analysis

target_order_id   → kept for later basket analysis

product_id        → kept to know which product was predicted

label             → 0 or 1, the answer to predict

features          → the 224 model inputs

## 4. Baseline Model — Logistic Regression

We first train a Logistic Regression model to establish a baseline.

The model learns the relationship between the engineered features and the probability that a product will be reordered.

This baseline will later be compared with more advanced machine learning models.

In [0]:
from pyspark.ml.classification import LogisticRegression

# Define the baseline Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=30,
    regParam=0.01
)

# Train the model
lr_model = lr.fit(train_ml_df)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


### 4.1 — Generate Validation Predictions

We apply the trained Logistic Regression model to the validation dataset.

For each customer-product pair, the model produces:

- `label` — the true outcome
- `reorder_probability` — estimated probability that the product will be reordered
- `prediction` — the final predicted class (`0` or `1`)

In [0]:
from pyspark.ml.functions import vector_to_array

# Generate predictions on validation data
lr_predictions_df = lr_model.transform(validation_ml_df)

# Extract the probability of class 1 = reordered
lr_predictions_df = (
    lr_predictions_df
    .withColumn(
        "reorder_probability",
        vector_to_array("probability")[1]
    )
)

# Display sample predictions
display(
    lr_predictions_df.select(
        "user_id",
        "product_id",
        "label",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "prediction"
    ).limit(10)
)

user_id,product_id,label,reorder_probability,prediction
61672,21903,0.0,0.0873,0.0
3830,27344,0.0,0.1966,0.0
147403,31596,0.0,0.0478,0.0
82750,34244,0.0,0.0669,0.0
155489,46667,0.0,0.0393,0.0
43783,10669,0.0,0.0735,0.0
92532,26346,0.0,0.4248,0.0
103343,588,0.0,0.4619,0.0
162988,21614,1.0,0.2422,0.0
123629,13517,1.0,0.1005,0.0


product 42768 → true label = 1

probability = 0.1090

prediction = 0

The model missed that reorder. This is why we need proper evaluation rather than judging a few examples.

### 4.2 — Evaluate ROC AUC and PR AUC

We evaluate the model using two ranking metrics:

- **ROC AUC** measures how well the model separates reordered from non-reordered products.
- **PR AUC** focuses more strongly on the positive class and is especially important because reorders represent only about 9.8% of our observations.

Higher values indicate better performance.

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# ROC AUC
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

roc_auc = roc_evaluator.evaluate(lr_predictions_df)

# PR AUC
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

pr_auc = pr_evaluator.evaluate(lr_predictions_df)

print("ROC AUC:", round(roc_auc, 4))
print("PR AUC:", round(pr_auc, 4))

ROC AUC: 0.8278
PR AUC: 0.4101


### 4.3 — Evaluate Classification Performance

ROC AUC and PR AUC evaluate how well the model ranks products by reorder probability.

We now evaluate the final `0` or `1` predictions using:

- **Precision** — among products predicted as reorders, how many were actually reordered?
- **Recall** — among products actually reordered, how many did the model detect?
- **F1-score** — balances Precision and Recall.

We also calculate the confusion matrix: True Positives, False Positives, True Negatives, and False Negatives.

In [0]:
# Calculate confusion matrix values
metrics = (
    lr_predictions_df
    .agg(
        F.sum(
            F.when((F.col("label") == 1) & (F.col("prediction") == 1), 1)
             .otherwise(0)
        ).alias("TP"),

        F.sum(
            F.when((F.col("label") == 0) & (F.col("prediction") == 1), 1)
             .otherwise(0)
        ).alias("FP"),

        F.sum(
            F.when((F.col("label") == 0) & (F.col("prediction") == 0), 1)
             .otherwise(0)
        ).alias("TN"),

        F.sum(
            F.when((F.col("label") == 1) & (F.col("prediction") == 0), 1)
             .otherwise(0)
        ).alias("FN")
    )
    .collect()[0]
)

tp = metrics["TP"]
fp = metrics["FP"]
tn = metrics["TN"]
fn = metrics["FN"]

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print("True Positives:", tp)
print("False Positives:", fp)
print("True Negatives:", tn)
print("False Negatives:", fn)

print("\nPrecision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

True Positives: 30326
False Positives: 18541
True Negatives: 1506362
False Negatives: 134223

Precision: 0.6206
Recall: 0.1843
F1-score: 0.2842


Precision = 61.79%

Recall    = 18.42%

F1-score  = 28.38%

So when the model predicts a reorder, it is often correct, but it detects only about 18% of the products that were actually reordered. For a recommendation system, that recall is probably too low.

This does not mean the model is bad. Our ROC AUC and PR AUC showed that its probability ranking is useful. The issue is that the default 0.50 cutoff may be too strict.

### 4.4 — Evaluate Different Classification Thresholds

The default Logistic Regression threshold is `0.50`.

Because reorders are relatively rare, this threshold may be too strict and can cause the model to miss many true reorders.

We therefore compare several probability thresholds and measure their Precision, Recall, and F1-score.

Lowering the threshold usually increases Recall but may decrease Precision.

In [0]:
# Thresholds to compare
thresholds_df = spark.createDataFrame(
    [(0.10,), (0.20,), (0.30,), (0.40,), (0.50,)],
    ["threshold"]
)

# Compare predictions at each threshold
threshold_metrics_df = (
    lr_predictions_df
    .select("label", "reorder_probability")
    .crossJoin(F.broadcast(thresholds_df))
    .withColumn(
        "threshold_prediction",
        F.when(
            F.col("reorder_probability") >= F.col("threshold"),
            1
        ).otherwise(0)
    )
    .groupBy("threshold")
    .agg(
        F.sum(
            F.when(
                (F.col("label") == 1) &
                (F.col("threshold_prediction") == 1),
                1
            ).otherwise(0)
        ).alias("TP"),

        F.sum(
            F.when(
                (F.col("label") == 0) &
                (F.col("threshold_prediction") == 1),
                1
            ).otherwise(0)
        ).alias("FP"),

        F.sum(
            F.when(
                (F.col("label") == 1) &
                (F.col("threshold_prediction") == 0),
                1
            ).otherwise(0)
        ).alias("FN")
    )
    .withColumn(
        "precision",
        F.round(
            F.col("TP") / (F.col("TP") + F.col("FP")),
            4
        )
    )
    .withColumn(
        "recall",
        F.round(
            F.col("TP") / (F.col("TP") + F.col("FN")),
            4
        )
    )
    .withColumn(
        "f1_score",
        F.round(
            2 * F.col("precision") * F.col("recall")
            / (F.col("precision") + F.col("recall")),
            4
        )
    )
    .select(
        "threshold",
        "precision",
        "recall",
        "f1_score"
    )
    .orderBy("threshold")
)

display(threshold_metrics_df)

threshold,precision,recall,f1_score
0.1,0.2644,0.7078,0.385
0.2,0.3927,0.4786,0.4314
0.3,0.4779,0.3558,0.4079
0.4,0.5527,0.2592,0.3529
0.5,0.6206,0.1843,0.2842


### 4.5 — Select the Classification Threshold

Among the tested thresholds, `0.20` produces the highest F1-score.

Compared with the default threshold of `0.50`, it substantially improves Recall while maintaining a reasonable Precision.

We therefore use `0.20` as the provisional classification threshold for the Logistic Regression baseline.

This threshold may later be reconsidered when we evaluate the system using recommendation-oriented Top-K metrics.

In [0]:
selected_threshold = 0.20

lr_predictions_tuned_df = (
    lr_predictions_df
    .withColumn(
        "tuned_prediction",
        F.when(
            F.col("reorder_probability") >= selected_threshold,
            1
        ).otherwise(0)
    )
)

display(
    lr_predictions_tuned_df.select(
        "user_id",
        "product_id",
        "label",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "tuned_prediction"
    ).limit(10)
)

user_id,product_id,label,reorder_probability,tuned_prediction
61672,21903,0.0,0.0873,0
3830,27344,0.0,0.1966,0
147403,31596,0.0,0.0478,0
82750,34244,0.0,0.0669,0
155489,46667,0.0,0.0393,0
43783,10669,0.0,0.0735,0
92532,26346,0.0,0.4248,1
103343,588,0.0,0.4619,1
162988,21614,1.0,0.2422,1
123629,13517,1.0,0.1005,0


## 5. Random Forest Model

We now train a Random Forest classifier as a more flexible alternative to Logistic Regression.

Unlike Logistic Regression, Random Forest can capture nonlinear relationships and interactions between customer, product, recency, timing, and basket-behavior features.

We will later compare both models using the same validation dataset and evaluation metrics.

In [0]:
from pyspark.ml.classification import RandomForestClassifier

# Define the Random Forest model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    numTrees=30,
    maxDepth=8,
    featureSubsetStrategy="sqrt",
    subsamplingRate=0.8,
    seed=42
)

# Train the model
rf_model = rf.fit(train_ml_df)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


### 5.1 — Generate Validation Predictions

We apply the trained Random Forest model to the same validation dataset used for Logistic Regression.

For each customer-product pair, we extract the predicted probability that the product will be reordered.

In [0]:
# Generate Random Forest predictions
rf_predictions_df = rf_model.transform(validation_ml_df)

# Extract probability of class 1 = reordered
rf_predictions_df = (
    rf_predictions_df
    .withColumn(
        "reorder_probability",
        vector_to_array("probability")[1]
    )
)

# Display sample predictions
display(
    rf_predictions_df.select(
        "user_id",
        "product_id",
        "label",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "prediction"
    ).limit(10)
)

user_id,product_id,label,reorder_probability,prediction
61672,21903,0.0,0.0495,0.0
3830,27344,0.0,0.1838,0.0
147403,31596,0.0,0.0548,0.0
82750,34244,0.0,0.0711,0.0
155489,46667,0.0,0.0498,0.0
43783,10669,0.0,0.1037,0.0
92532,26346,0.0,0.2489,0.0
103343,588,0.0,0.4629,0.0
162988,21614,1.0,0.2219,0.0
123629,13517,1.0,0.0739,0.0


### 5.2 — Evaluate Random Forest Performance

We evaluate the Random Forest using the same ranking metrics used for Logistic Regression:

- **ROC AUC** — ability to distinguish reordered from non-reordered products.
- **PR AUC** — performance on the positive reorder class, which is especially important because the dataset is imbalanced.

Using the same validation data and metrics allows a fair comparison between the two models.

In [0]:
# Evaluate Random Forest ROC AUC
rf_roc_auc = roc_evaluator.evaluate(rf_predictions_df)

# Evaluate Random Forest PR AUC
rf_pr_auc = pr_evaluator.evaluate(rf_predictions_df)

print("Random Forest ROC AUC:", round(rf_roc_auc, 4))
print("Random Forest PR AUC:", round(rf_pr_auc, 4))

Random Forest ROC AUC: 0.8163
Random Forest PR AUC: 0.4034


### 5.3 — Evaluate Random Forest Classification Thresholds

The default Random Forest threshold of `0.50` may not be appropriate for our imbalanced dataset.

We therefore test several probability thresholds and compare Precision, Recall, and F1-score.

This allows us to identify the threshold that gives the best balance between correctly recommending reordered products and avoiding false recommendations.

In [0]:
# Thresholds to evaluate
rf_thresholds_df = spark.createDataFrame(
    [
        (0.05,),
        (0.10,),
        (0.15,),
        (0.20,),
        (0.25,),
        (0.30,)
    ],
    ["threshold"]
)

# Evaluate Random Forest at each threshold
rf_threshold_metrics_df = (
    rf_predictions_df
    .select("label", "reorder_probability")
    .crossJoin(F.broadcast(rf_thresholds_df))
    .withColumn(
        "threshold_prediction",
        F.when(
            F.col("reorder_probability") >= F.col("threshold"),
            1
        ).otherwise(0)
    )
    .groupBy("threshold")
    .agg(
        F.sum(
            F.when(
                (F.col("label") == 1) &
                (F.col("threshold_prediction") == 1),
                1
            ).otherwise(0)
        ).alias("TP"),

        F.sum(
            F.when(
                (F.col("label") == 0) &
                (F.col("threshold_prediction") == 1),
                1
            ).otherwise(0)
        ).alias("FP"),

        F.sum(
            F.when(
                (F.col("label") == 1) &
                (F.col("threshold_prediction") == 0),
                1
            ).otherwise(0)
        ).alias("FN")
    )
    .withColumn(
        "precision",
        F.round(
            F.col("TP") / (F.col("TP") + F.col("FP")),
            4
        )
    )
    .withColumn(
        "recall",
        F.round(
            F.col("TP") / (F.col("TP") + F.col("FN")),
            4
        )
    )
    .withColumn(
        "f1_score",
        F.round(
            2 * F.col("precision") * F.col("recall")
            / (F.col("precision") + F.col("recall")),
            4
        )
    )
    .select(
        "threshold",
        "precision",
        "recall",
        "f1_score"
    )
    .orderBy("threshold")
)

display(rf_threshold_metrics_df)

threshold,precision,recall,f1_score
0.05,0.1455,0.9253,0.2515
0.1,0.2704,0.6714,0.3855
0.15,0.3562,0.5261,0.4248
0.2,0.4127,0.4414,0.4266
0.25,0.4588,0.3705,0.4099
0.3,0.5165,0.2963,0.3766


### 5.4 — Compare Logistic Regression and Random Forest

We compare both models using the same validation dataset.

The comparison includes:

- ROC AUC
- PR AUC
- Best classification threshold
- Precision
- Recall
- F1-score

The best model should provide strong ranking performance while maintaining a useful balance between Precision and Recall.

In [0]:
model_comparison_df = spark.createDataFrame(
    [
        (
            "Logistic Regression",
            0.8270,
            0.4093,
            0.20,
            0.3912,
            0.4796,
            0.4309
        ),
        (
            "Random Forest",
            0.8163,
            0.4034,
            0.20,
            0.4127,
            0.4414,
            0.4266
        )
    ],
    [
        "model",
        "roc_auc",
        "pr_auc",
        "best_threshold",
        "precision",
        "recall",
        "f1_score"
    ]
)

display(model_comparison_df)

model,roc_auc,pr_auc,best_threshold,precision,recall,f1_score
Logistic Regression,0.827,0.4093,0.2,0.3912,0.4796,0.4309
Random Forest,0.8163,0.4034,0.2,0.4127,0.4414,0.4266


## 6. Gradient-Boosted Trees Model

We now train a Gradient-Boosted Trees classifier.

Unlike Random Forest, where trees are built independently, Gradient-Boosted Trees build trees sequentially. Each new tree focuses on correcting errors made by the previous trees.

This allows the model to capture complex nonlinear relationships between customer behavior, product history, recency, basket context, and temporal features.

In [0]:
from pyspark.ml.classification import GBTClassifier

# Define the Gradient-Boosted Trees model
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=20,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42
)

# Train the model
gbt_model = gbt.fit(train_ml_df)

print("Gradient-Boosted Trees model trained successfully.")

Gradient-Boosted Trees model trained successfully.


### 6.1 — Generate Validation Predictions

We apply the trained Gradient-Boosted Trees model to the same validation dataset.

For each customer-product pair, we extract the predicted probability that the product will be reordered.

In [0]:
# Generate GBT predictions
gbt_predictions_df = gbt_model.transform(validation_ml_df)

# Extract probability of class 1 = reordered
gbt_predictions_df = (
    gbt_predictions_df
    .withColumn(
        "reorder_probability",
        vector_to_array("probability")[1]
    )
)

# Display sample predictions
display(
    gbt_predictions_df.select(
        "user_id",
        "product_id",
        "label",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "prediction"
    ).limit(10)
)

user_id,product_id,label,reorder_probability,prediction
61672,21903,0.0,0.0925,0.0
3830,27344,0.0,0.1493,0.0
147403,31596,0.0,0.0756,0.0
82750,34244,0.0,0.0892,0.0
155489,46667,0.0,0.0684,0.0
43783,10669,0.0,0.109,0.0
92532,26346,0.0,0.4125,0.0
103343,588,0.0,0.4156,0.0
162988,21614,1.0,0.3111,0.0
123629,13517,1.0,0.1057,0.0


### 6.2 — Evaluate Gradient-Boosted Trees Performance

We evaluate the Gradient-Boosted Trees model using the same ROC AUC and PR AUC metrics used for the previous models.

Using the same validation dataset and evaluation metrics allows us to compare all models fairly.

In [0]:
# Evaluate GBT ROC AUC
gbt_roc_auc = roc_evaluator.evaluate(gbt_predictions_df)

# Evaluate GBT PR AUC
gbt_pr_auc = pr_evaluator.evaluate(gbt_predictions_df)

print("GBT ROC AUC:", round(gbt_roc_auc, 4))
print("GBT PR AUC:", round(gbt_pr_auc, 4))

GBT ROC AUC: 0.8295
GBT PR AUC: 0.4171


### 6.3 — Evaluate GBT Classification Thresholds

The default threshold of `0.50` may be too strict for our imbalanced reorder prediction problem.

We test several probability thresholds and compare Precision, Recall, and F1-score to identify the best balance for the GBT model.

In [0]:
gbt_thresholds_df = spark.createDataFrame(
    [
        (0.05,),
        (0.10,),
        (0.15,),
        (0.20,),
        (0.25,),
        (0.30,),
        (0.35,),
        (0.40,)
    ],
    ["threshold"]
)

gbt_threshold_metrics_df = (
    gbt_predictions_df
    .select("label", "reorder_probability")
    .crossJoin(F.broadcast(gbt_thresholds_df))
    .withColumn(
        "threshold_prediction",
        F.when(
            F.col("reorder_probability") >= F.col("threshold"),
            1
        ).otherwise(0)
    )
    .groupBy("threshold")
    .agg(
        F.sum(
            F.when(
                (F.col("label") == 1) &
                (F.col("threshold_prediction") == 1),
                1
            ).otherwise(0)
        ).alias("TP"),

        F.sum(
            F.when(
                (F.col("label") == 0) &
                (F.col("threshold_prediction") == 1),
                1
            ).otherwise(0)
        ).alias("FP"),

        F.sum(
            F.when(
                (F.col("label") == 1) &
                (F.col("threshold_prediction") == 0),
                1
            ).otherwise(0)
        ).alias("FN")
    )
    .withColumn(
        "precision",
        F.round(
            F.col("TP") / (F.col("TP") + F.col("FP")),
            4
        )
    )
    .withColumn(
        "recall",
        F.round(
            F.col("TP") / (F.col("TP") + F.col("FN")),
            4
        )
    )
    .withColumn(
        "f1_score",
        F.round(
            2 * F.col("precision") * F.col("recall")
            / (F.col("precision") + F.col("recall")),
            4
        )
    )
    .select(
        "threshold",
        "precision",
        "recall",
        "f1_score"
    )
    .orderBy("threshold")
)

display(gbt_threshold_metrics_df)

threshold,precision,recall,f1_score
0.05,0.1099,0.9904,0.1978
0.1,0.2338,0.7728,0.359
0.15,0.3129,0.6265,0.4174
0.2,0.3796,0.5177,0.438
0.25,0.4337,0.4325,0.4331
0.3,0.4778,0.3708,0.4176
0.35,0.522,0.3115,0.3902
0.4,0.5678,0.2517,0.3488


### 6.4 — Final Model Comparison

We compare the three trained models using the same validation dataset.

The comparison includes:

- ROC AUC
- PR AUC
- Best classification threshold
- Precision
- Recall
- F1-score

Among the tested models, Gradient-Boosted Trees provides the strongest overall performance.

In [0]:
final_model_comparison_df = spark.createDataFrame(
    [
        (
            "Logistic Regression",
            0.8270,
            0.4093,
            0.20,
            0.3912,
            0.4796,
            0.4309
        ),
        (
            "Random Forest",
            0.8163,
            0.4034,
            0.20,
            0.4127,
            0.4414,
            0.4266
        ),
        (
            "Gradient-Boosted Trees",
            0.8295,
            0.4171,
            0.20,
            0.3796,
            0.5177,
            0.4380
        )
    ],
    [
        "model",
        "roc_auc",
        "pr_auc",
        "best_threshold",
        "precision",
        "recall",
        "f1_score"
    ]
)

display(
    final_model_comparison_df
    .orderBy(F.desc("f1_score"))
)

model,roc_auc,pr_auc,best_threshold,precision,recall,f1_score
Gradient-Boosted Trees,0.8295,0.4171,0.2,0.3796,0.5177,0.438
Logistic Regression,0.827,0.4093,0.2,0.3912,0.4796,0.4309
Random Forest,0.8163,0.4034,0.2,0.4127,0.4414,0.4266


### 6.5 — Select the Champion Model

Gradient-Boosted Trees is selected as the current champion model because it achieves the strongest overall validation performance.

Its results are:

- ROC AUC: `0.8295`
- PR AUC: `0.4171`
- Best threshold: `0.20`
- Precision: `0.3796`
- Recall: `0.5177`
- F1-score: `0.4380`

The model provides the best combination of ranking quality and reorder detection among the three tested algorithms.

However, classification metrics alone do not fully evaluate a recommendation system. We will next evaluate how well the model ranks products within each customer's candidate basket using recommendation-oriented metrics.

In [0]:
champion_model_name = "Gradient-Boosted Trees"
champion_model = gbt_model
champion_predictions_df = gbt_predictions_df
champion_threshold = 0.20

print("Champion model:", champion_model_name)
print("Selected threshold:", champion_threshold)

Champion model: Gradient-Boosted Trees
Selected threshold: 0.2


## 7. Recommendation-Oriented Evaluation

Classification metrics evaluate each customer-product pair independently.

However, the final objective is to recommend the products that are most likely to appear in each customer's next basket.

We therefore rank each customer's candidate products by predicted reorder probability.

A rank of `1` represents the product that the model considers most likely to be reordered.

In [0]:
from pyspark.sql.window import Window

# Define ranking within each customer
recommendation_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.desc("reorder_probability"),
        F.asc("product_id")
    )
)

# Rank candidate products for every validation customer
gbt_ranked_df = (
    champion_predictions_df
    .withColumn(
        "recommendation_rank",
        F.row_number().over(recommendation_window)
    )
)

# Display the top 5 recommendations for a few customers
display(
    gbt_ranked_df
    .filter(F.col("recommendation_rank") <= 5)
    .select(
        "user_id",
        "product_id",
        "label",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "recommendation_rank"
    )
    .orderBy(
        "user_id",
        "recommendation_rank"
    )
    .limit(25)
)

user_id,product_id,label,reorder_probability,recommendation_rank
2,47209,0.0,0.467,1
2,1559,0.0,0.4133,2
2,18523,0.0,0.4133,3
2,19156,0.0,0.4133,4
2,24852,1.0,0.3808,5
18,36216,1.0,0.5224,1
18,47546,1.0,0.37,2
18,10807,0.0,0.1617,3
18,21137,1.0,0.161,4
18,27729,0.0,0.1478,5


In [0]:
display(
    gbt_ranked_df
    .filter(F.col("recommendation_rank") <= 5)
    .select(
        "user_id",
        "product_id",
        "label",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "recommendation_rank"
    )
    .orderBy(
        F.asc("user_id"),
        F.asc("recommendation_rank")
    )
    .limit(25)
)

user_id,product_id,label,reorder_probability,recommendation_rank
2,47209,0.0,0.467,1
2,1559,0.0,0.4133,2
2,18523,0.0,0.4133,3
2,19156,0.0,0.4133,4
2,24852,1.0,0.3808,5
18,36216,1.0,0.5224,1
18,47546,1.0,0.37,2
18,10807,0.0,0.1617,3
18,21137,1.0,0.161,4
18,27729,0.0,0.1478,5


### 7.2 — Precision@5

Precision@5 measures how many of the products recommended in the customer's top 5 were actually reordered.

For each customer, we calculate:

**Precision@5 = Relevant products in Top 5 / Number of products recommended**

A higher value means the recommendation list contains more products that actually appeared in the customer's next basket.

In [0]:
# Keep the top 5 recommendations for each customer
top5_df = (
    gbt_ranked_df
    .filter(F.col("recommendation_rank") <= 5)
)

# Calculate Precision@5 for each customer
precision_at_5_df = (
    top5_df
    .groupBy("user_id")
    .agg(
        F.sum("label").alias("relevant_in_top5"),
        F.count("*").alias("recommended_products")
    )
    .withColumn(
        "precision_at_5",
        F.col("relevant_in_top5") / F.col("recommended_products")
    )
)

# Calculate average Precision@5 across validation customers
average_precision_at_5 = (
    precision_at_5_df
    .agg(F.avg("precision_at_5").alias("avg_precision_at_5"))
    .collect()[0]["avg_precision_at_5"]
)

print("Average Precision@5:", round(average_precision_at_5, 4))

Average Precision@5: 0.386


### 7.3 — Recall@5

Recall@5 measures how many of the products that were actually reordered appear within the model's Top 5 recommendations.

For each customer:

**Recall@5 = Relevant products in Top 5 / Total relevant candidate products**

Because our current candidate-generation strategy only includes products previously purchased by the customer, this metric evaluates recall within the available candidate set.

Customers with no relevant products in the candidate set are excluded because Recall is undefined for them.

In [0]:
# Count all relevant candidate products for each customer
relevant_products_df = (
    gbt_ranked_df
    .groupBy("user_id")
    .agg(
        F.sum("label").alias("total_relevant_products")
    )
)

# Calculate Recall@5 for each customer
recall_at_5_df = (
    precision_at_5_df
    .select(
        "user_id",
        "relevant_in_top5"
    )
    .join(
        relevant_products_df,
        on="user_id",
        how="inner"
    )
    .filter(
        F.col("total_relevant_products") > 0
    )
    .withColumn(
        "recall_at_5",
        F.col("relevant_in_top5")
        / F.col("total_relevant_products")
    )
)

# Average Recall@5
average_recall_at_5 = (
    recall_at_5_df
    .agg(
        F.avg("recall_at_5").alias("avg_recall_at_5")
    )
    .collect()[0]["avg_recall_at_5"]
)

print("Average Recall@5:", round(average_recall_at_5, 4))

Average Recall@5: 0.4052


### 7.4 — Hit Rate@5

Hit Rate@5 measures the proportion of customers for whom at least one actually reordered product appears in the Top 5 recommendations.

For each customer:

- `1` — at least one Top-5 recommendation was actually reordered
- `0` — none of the Top-5 recommendations were reordered

A higher Hit Rate means the recommendation system is useful for a larger proportion of customers.

In [0]:
# Determine whether each customer has at least one correct recommendation
hit_rate_at_5_df = (
    top5_df
    .groupBy("user_id")
    .agg(
        F.max("label").alias("hit_at_5")
    )
)

# Calculate average Hit Rate@5
hit_rate_at_5 = (
    hit_rate_at_5_df
    .agg(
        F.avg("hit_at_5").alias("hit_rate_at_5")
    )
    .collect()[0]["hit_rate_at_5"]
)

print("Hit Rate@5:", round(hit_rate_at_5, 4))

Hit Rate@5: 0.8044


### 7.5 — NDCG@5

NDCG@5 evaluates the quality of the ranking itself.

Correct recommendations receive more credit when they appear near the top of the Top-5 list.

For example, an actually reordered product at rank 1 contributes more than the same product at rank 5.

NDCG ranges from:

- `0` — poor ranking
- `1` — ideal ranking

Customers with no relevant products in the available candidate set are excluded because NDCG is not defined for them.

In [0]:
# Calculate DCG@5 for each customer
dcg_at_5_df = (
    top5_df
    .withColumn(
        "discounted_gain",
        F.col("label") / F.log2(F.col("recommendation_rank") + 1)
    )
    .groupBy("user_id")
    .agg(
        F.sum("discounted_gain").alias("dcg_at_5")
    )
)

# Keep customers who have at least one relevant candidate product
eligible_customers_df = (
    relevant_products_df
    .filter(F.col("total_relevant_products") > 0)
)

# Create ideal ranking positions 1 to 5
ideal_ranks_df = (
    spark.range(1, 6)
    .withColumnRenamed("id", "ideal_rank")
)

# Calculate the best possible DCG@5 for each customer
idcg_at_5_df = (
    eligible_customers_df
    .crossJoin(F.broadcast(ideal_ranks_df))
    .withColumn(
        "ideal_gain",
        F.when(
            F.col("ideal_rank") <= F.least(
                F.col("total_relevant_products"),
                F.lit(5)
            ),
            1 / F.log2(F.col("ideal_rank") + 1)
        ).otherwise(0)
    )
    .groupBy("user_id")
    .agg(
        F.sum("ideal_gain").alias("idcg_at_5")
    )
)

# Calculate NDCG@5
ndcg_at_5_df = (
    dcg_at_5_df
    .join(idcg_at_5_df, on="user_id", how="inner")
    .withColumn(
        "ndcg_at_5",
        F.col("dcg_at_5") / F.col("idcg_at_5")
    )
)

# Average NDCG@5 across eligible validation customers
average_ndcg_at_5 = (
    ndcg_at_5_df
    .agg(
        F.avg("ndcg_at_5").alias("avg_ndcg_at_5")
    )
    .collect()[0]["avg_ndcg_at_5"]
)

print("Average NDCG@5:", round(average_ndcg_at_5, 4))

Average NDCG@5: 0.5259


### 7.6 — MAP@5

Mean Average Precision at 5 (MAP@5) evaluates both recommendation accuracy and ranking order.

For each customer, correct recommendations receive more credit when they appear earlier in the Top-5 list.

The Average Precision is calculated for each customer and then averaged across eligible validation customers.

A higher MAP@5 indicates a better-ranked recommendation list.

In [0]:
# Calculate cumulative number of relevant products up to each rank
map_window = (
    Window
    .partitionBy("user_id")
    .orderBy("recommendation_rank")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

map_at_5_rows_df = (
    top5_df
    .withColumn(
        "cumulative_relevant",
        F.sum("label").over(map_window)
    )
    .withColumn(
        "precision_at_rank",
        F.col("cumulative_relevant") / F.col("recommendation_rank")
    )
    .withColumn(
        "precision_contribution",
        F.when(
            F.col("label") == 1,
            F.col("precision_at_rank")
        ).otherwise(0)
    )
)

# Calculate Average Precision@5 for each eligible customer
average_precision_at_5_df = (
    map_at_5_rows_df
    .groupBy("user_id")
    .agg(
        F.sum("precision_contribution").alias("precision_sum")
    )
    .join(
        relevant_products_df,
        on="user_id",
        how="inner"
    )
    .filter(F.col("total_relevant_products") > 0)
    .withColumn(
        "ap_at_5",
        F.col("precision_sum")
        / F.least(F.col("total_relevant_products"), F.lit(5))
    )
)

# Calculate MAP@5
map_at_5 = (
    average_precision_at_5_df
    .agg(
        F.avg("ap_at_5").alias("map_at_5")
    )
    .collect()[0]["map_at_5"]
)

print("MAP@5:", round(map_at_5, 4))

MAP@5: 0.4248


### 7.7 — Recommendation Performance Summary

We summarize the main Top-5 recommendation metrics for the champion Gradient-Boosted Trees model.

These metrics evaluate different aspects of recommendation quality:

- **Precision@5** — accuracy of the Top-5 recommendations.
- **Recall@5** — proportion of relevant candidate products recovered.
- **Hit Rate@5** — proportion of customers receiving at least one correct recommendation.
- **NDCG@5** — quality of the ranking, giving more importance to correct products near the top.
- **MAP@5** — overall precision and ranking quality across customers.

In [0]:
recommendation_metrics_df = spark.createDataFrame(
    [
        ("Precision@5", round(average_precision_at_5, 4)),
        ("Recall@5", round(average_recall_at_5, 4)),
        ("Hit Rate@5", round(hit_rate_at_5, 4)),
        ("NDCG@5", round(average_ndcg_at_5, 4)),
        ("MAP@5", round(map_at_5, 4))
    ],
    ["metric", "value"]
)

display(recommendation_metrics_df)

metric,value
Precision@5,0.386
Recall@5,0.4052
Hit Rate@5,0.8044
NDCG@5,0.5259
MAP@5,0.4248


## 8. Feature Importance

To understand what drives the Gradient-Boosted Trees model, we examine its feature importance scores.

Feature importance indicates how much each input feature contributes to the model's prediction decisions.

Higher importance means the feature played a larger role in distinguishing products that are likely to be reordered from those that are not.

In [0]:
# Get the sizes of the encoded categorical vectors
aisle_ohe_size = (
    train_encoded_df
    .schema["aisle_ohe"]
    .metadata["ml_attr"]["num_attrs"]
)

department_ohe_size = (
    train_encoded_df
    .schema["department_ohe"]
    .metadata["ml_attr"]["num_attrs"]
)

# Create a name for every position in the final feature vector
feature_names = (
    numerical_columns
    + [f"aisle_ohe_{i}" for i in range(aisle_ohe_size)]
    + [f"department_ohe_{i}" for i in range(department_ohe_size)]
)

# Extract importance scores from the champion GBT model
feature_importances = gbt_model.featureImportances.toArray()

print("Feature names:", len(feature_names))
print("Model importances:", len(feature_importances))

# Create a table of feature importances
feature_importance_df = spark.createDataFrame(
    [
        (name, float(importance))
        for name, importance in zip(
            feature_names,
            feature_importances
        )
    ],
    ["feature", "importance"]
)

# Show the 20 most important features
display(
    feature_importance_df
    .orderBy(F.desc("importance"))
    .limit(20)
)

Feature names: 224
Model importances: 224


feature,importance
user_product_order_share,0.26461854767184073
orders_since_last_product_purchase,0.19794186026061658
purchases_last_5_orders,0.1354946370877649
product_reorder_rate,0.06717145419782432
target_days_since_prior_order,0.04634712067606632
user_product_due_score,0.04209658743830959
purchase_rate_last_5_orders,0.039018378787068056
product_purchases_per_customer,0.03551197055327396
repeat_stability_signal,0.02516367724934629
customer_reorder_rate,0.021496501794777953


### 8.1 — Interpretation of the Most Important Features

The feature importance analysis shows that reorder prediction is driven primarily by the customer's historical relationship with each product.

The strongest features are:

- **`user_product_order_share`** — the proportion of a customer's previous orders containing the product. Products purchased consistently by the same customer are much more informative for predicting future reorders.

- **`orders_since_last_product_purchase`** — measures product recency. How long it has been since the customer last purchased the product is a major signal of whether it may be due for another purchase.

- **`purchases_last_5_orders`** — captures recent purchasing frequency. Products repeatedly purchased in recent orders are strong reorder candidates.

Other important signals include:

- **`product_reorder_rate`** — some products naturally generate more repeat purchases across customers.
- **`target_days_since_prior_order`** — the timing of the customer's next shopping session influences reorder behavior.
- **`user_product_due_score`** — compares current product recency with the customer's historical repurchase cycle.
- **`purchase_rate_last_5_orders`** — measures recent customer-product consistency.
- **`repeat_stability_signal`** — combines recent basket repetition with the customer's overall basket stability.

Overall, the model indicates that **personal purchase frequency, recency, and repurchase cycles are more influential than general product popularity or broad category information**.

Feature importance describes how much the GBT model uses each feature for its decisions; it does not imply that the features causally determine reorder behavior.

## 9. Create the Final Recommendation Output

We convert the champion model's predictions into a practical Top-5 recommendation table.

For each validation customer, the table contains the five products with the highest predicted reorder probabilities, together with their recommendation rank and actual outcome.

This table represents the final output of the next-basket reorder recommendation model.

In [0]:
# Product information
product_names_df = (
    features_df
    .select(
        "product_id",
        "product_name"
    )
    .distinct()
)

# Final Top-5 recommendations
final_recommendations_df = (
    gbt_ranked_df
    .filter(F.col("recommendation_rank") <= 5)
    .select(
        "user_id",
        "target_order_id",
        "product_id",
        "label",
        "reorder_probability",
        "recommendation_rank"
    )
    .join(
        product_names_df,
        on="product_id",
        how="left"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id",
        "product_name",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "recommendation_rank",
        "label"
    )
)

display(
    final_recommendations_df
    .orderBy(
        "user_id",
        "recommendation_rank"
    )
    .limit(25)
)

user_id,target_order_id,product_id,product_name,reorder_probability,recommendation_rank,label
2,1492625,47209,Organic Hass Avocado,0.467,1,0.0
2,1492625,1559,Cherry Pomegranate Greek Yogurt,0.4133,2,0.0
2,1492625,18523,Total 2% All Natural Greek Strained Yogurt with Honey,0.4133,3,0.0
2,1492625,19156,Fat Free Blueberry Yogurt,0.4133,4,0.0
2,1492625,24852,Banana,0.3808,5,1.0
18,2461523,36216,Lime Italian Sparkling Mineral Water,0.5224,1,1.0
18,2461523,47546,Chocolate Coconut Milk Beverage,0.37,2,1.0
18,2461523,10807,Wild Arugula Salad,0.1617,3,0.0
18,2461523,21137,Organic Strawberries,0.161,4,1.0
18,2461523,27729,Cold-Pressed Organic Orange,0.1478,5,0.0


### 9.1 — Save the Top-5 Recommendation Results

We save the final Top-5 recommendations for the validation customers as a Delta table.

The table contains the recommended products, predicted reorder probabilities, ranking positions, and actual outcomes for evaluation and future analysis.

In [0]:
(
    final_recommendations_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.ml_data.top5_recommendations_validation")
)

print("Recommendation table saved successfully:")
print("workspace.ml_data.top5_recommendations_validation")

Recommendation table saved successfully:
workspace.ml_data.top5_recommendations_validation


### 9.2 — Build the Reusable Champion Pipeline

The GBT model expects already-prepared feature vectors.

To make the solution reusable, we combine the fitted categorical preprocessing, feature assembler, and champion GBT model into one scoring pipeline.

This pipeline can later transform raw engineered features directly into reorder predictions.

In [0]:
from pyspark.ml import PipelineModel

champion_pipeline_model = PipelineModel(
    stages=
        list(categorical_model.stages)
        + [assembler, gbt_model]
)

print("Reusable champion pipeline created successfully.")
print("Number of pipeline stages:", len(champion_pipeline_model.stages))

Reusable champion pipeline created successfully.
Number of pipeline stages: 5


### 9.3 — Validate the Complete Scoring Pipeline

Before saving the model, we verify that the complete pipeline can transform raw engineered features directly into reorder predictions.

This confirms that categorical encoding, feature assembly, and GBT prediction work together as one reusable scoring process.

In [0]:
# Test the complete pipeline on a small sample
pipeline_test_df = champion_pipeline_model.transform(
    validation_df.limit(10)
)

# Extract reorder probability
pipeline_test_df = (
    pipeline_test_df
    .withColumn(
        "reorder_probability",
        vector_to_array("probability")[1]
    )
)

display(
    pipeline_test_df.select(
        "user_id",
        "product_id",
        "target_reordered",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "prediction"
    )
)

user_id,product_id,target_reordered,reorder_probability,prediction
61672,21903,0,0.0925,0.0
3830,27344,0,0.1493,0.0
147403,31596,0,0.0756,0.0
82750,34244,0,0.0892,0.0
155489,46667,0,0.0684,0.0
43783,10669,0,0.109,0.0
92532,26346,0,0.4125,0.0
103343,588,0,0.4156,0.0
162988,21614,1,0.3111,0.0
123629,13517,1,0.1057,0.0


### 9.4 — Apply the Selected Classification Threshold

The champion GBT model achieved its best F1-score using a reorder probability threshold of `0.20`.

Before saving the reusable pipeline, we embed this selected threshold so that future `prediction` values use the validated decision rule rather than the default cutoff.

In [0]:
# Create a tuned copy of the fitted GBT model
gbt_model_tuned = gbt_model.copy({})

# Apply the selected 0.20 probability threshold
gbt_model_tuned.setThresholds([0.8, 0.2])

# Rebuild the reusable pipeline with the tuned GBT model
champion_pipeline_model = PipelineModel(
    stages=
        list(categorical_model.stages)
        + [assembler, gbt_model_tuned]
)

print("Champion threshold:", champion_threshold)
print("GBT thresholds:", gbt_model_tuned.getThresholds())
print("Reusable pipeline updated successfully.")

Champion threshold: 0.2
GBT thresholds: [0.8, 0.2]
Reusable pipeline updated successfully.


### 9.5 — Save the Champion Model Pipeline

We save the complete champion pipeline, including categorical preprocessing, feature assembly, the trained Gradient-Boosted Trees model, and the selected classification threshold.

This allows the trained recommendation model to be reused later without retraining it.

In [0]:
# Create a Unity Catalog Volume for saved ML models
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.ml_data.models
""")

# Path for the champion pipeline
champion_model_path = (
    "/Volumes/workspace/ml_data/models/"
    "instacart_gbt_reorder_pipeline"
)

# Save the complete reusable pipeline
(
    champion_pipeline_model
    .write()
    .overwrite()
    .save(champion_model_path)
)

print("Champion pipeline saved successfully.")
print("Model path:", champion_model_path)

Champion pipeline saved successfully.
Model path: /Volumes/workspace/ml_data/models/instacart_gbt_reorder_pipeline


### 9.6 — Reload and Verify the Saved Pipeline

We reload the saved champion pipeline from the Unity Catalog Volume and test it on validation data.

This confirms that the complete trained model can be reused in a future notebook or application without retraining.

In [0]:
from pyspark.ml import PipelineModel

# Reload the saved champion pipeline
loaded_champion_pipeline = PipelineModel.load(
    champion_model_path
)

# Test the reloaded pipeline
loaded_test_df = loaded_champion_pipeline.transform(
    validation_df.limit(10)
)

loaded_test_df = (
    loaded_test_df
    .withColumn(
        "reorder_probability",
        vector_to_array("probability")[1]
    )
)

display(
    loaded_test_df.select(
        "user_id",
        "product_id",
        "target_reordered",
        F.round("reorder_probability", 4).alias("reorder_probability"),
        "prediction"
    )
)

user_id,product_id,target_reordered,reorder_probability,prediction
61672,21903,0,0.0925,0.0
3830,27344,0,0.1493,0.0
147403,31596,0,0.0756,0.0
82750,34244,0,0.0892,0.0
155489,46667,0,0.0684,0.0
43783,10669,0,0.109,0.0
92532,26346,0,0.4125,1.0
103343,588,0,0.4156,1.0
162988,21614,1,0.3111,1.0
123629,13517,1,0.1057,0.0


## 10. Conclusion

In this notebook, we developed and evaluated a machine learning system for predicting product reorders and generating personalized next-basket recommendations.

Three classification models were compared:

- Logistic Regression
- Random Forest
- Gradient-Boosted Trees

Gradient-Boosted Trees achieved the strongest overall validation performance:

- **ROC AUC:** 0.8295
- **PR AUC:** 0.4171
- **Precision:** 0.3796
- **Recall:** 0.5177
- **F1-score:** 0.4380
- **Selected probability threshold:** 0.20

The model was also evaluated as a recommendation system using Top-5 ranking metrics:

- **Precision@5:** 0.3860
- **Recall@5:** 0.4052
- **Hit Rate@5:** 0.8044
- **NDCG@5:** 0.5259
- **MAP@5:** 0.4248

Feature-importance analysis showed that reorder behavior is driven mainly by customer-product purchase frequency, recency, recent purchase activity, and repurchase cycles.

The final Gradient-Boosted Trees pipeline was saved with its preprocessing steps and selected classification threshold, and successfully reloaded for inference.

### Current Limitation

The current candidate-generation strategy considers only products previously purchased by each customer. Earlier analysis showed that this baseline candidate set covers approximately **59.86% of products appearing in the target baskets**.

Therefore, the current system is primarily a **reorder recommendation model** and cannot yet recommend new-to-customer products.

A future extension can improve candidate generation using category affinity, product popularity, and co-purchase relationships to expand the system toward full next-basket prediction.